In [9]:
# ==========================================
# IMPORTS
# ==========================================

import pandas as pd
import numpy as np
from pathlib import Path

# Configuración pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Librerías cargadas")

✅ Librerías cargadas


In [10]:
# ==========================================
# PATHS
# ==========================================

DATA_DIR = Path("../data/raw")

CSV_FILE = DATA_DIR / "XEMA-Museu_Badalona_Dades_meteorològiques_de_la_XEMA_20260614.csv"

print(CSV_FILE)

../data/raw/XEMA-Museu_Badalona_Dades_meteorològiques_de_la_XEMA_20260614.csv


In [11]:
# ==========================================
# LOAD CSV
# ==========================================

df = pd.read_csv(
    CSV_FILE,
    sep=",",
    low_memory=False
)

print("Shape:")
print(df.shape)

print("\nColumnas:")
print(df.columns.tolist())

Shape:
(4791830, 8)

Columnas:
['ID', 'CODI_ESTACIO', 'CODI_VARIABLE', 'DATA_LECTURA', 'DATA_EXTREM', 'VALOR_LECTURA', 'CODI_ESTAT', 'CODI_BASE']


In [12]:
# ==========================================
# QUICK INSPECTION
# ==========================================

display(df.head())

print("\nInfo dataframe:\n")
print(df.info())

print("\nMemoria utilizada:")
print(
    round(df.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB"
)

,ID,CODI_ESTACIO,CODI_VARIABLE,DATA_LECTURA,DATA_EXTREM,VALOR_LECTURA,CODI_ESTAT,CODI_BASE
0,WU320101090000,WU,32,01/01/2009 12:00:00 AM,NaN,"10,2",V,SH
1,WU330101090000,WU,33,01/01/2009 12:00:00 AM,NaN,89,V,SH
2,WU340101090000,WU,34,01/01/2009 12:00:00 AM,NaN,1.016,V,SH
3,WU350101090000,WU,35,01/01/2009 12:00:00 AM,NaN,0,V,SH
4,WU010101090000,WU,1,01/01/2009 12:00:00 AM,01/01/2009 12:00:00 AM,1.017,V,HO



Info dataframe:

<class 'pandas.DataFrame'>
RangeIndex: 4791830 entries, 0 to 4791829
Data columns (total 8 columns):
 #   Column         Dtype
---  ------         -----
 0   ID             str  
 1   CODI_ESTACIO   str  
 2   CODI_VARIABLE  int64
 3   DATA_LECTURA   str  
 4   DATA_EXTREM    str  
 5   VALOR_LECTURA  str  
 6   CODI_ESTAT     str  
 7   CODI_BASE      str  
dtypes: int64(1), str(7)
memory usage: 545.4 MB
None

Memoria utilizada:
545.39 MB


In [13]:
# ==========================================
# MEMORY OPTIMIZATION
# ==========================================

print("Memoria inicial:")
print(
    round(df.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB"
)

# ------------------------------------------
# DATETIME
# ------------------------------------------

df["DATA_LECTURA"] = pd.to_datetime(
    df["DATA_LECTURA"],
    format="%d/%m/%Y %I:%M:%S %p",
    errors="coerce"
)


# ------------------------------------------
# DOWNCAST NUMERIC TYPES
# ------------------------------------------

df["CODI_VARIABLE"] = pd.to_numeric(
    df["CODI_VARIABLE"],
    downcast="integer"
)

# ------------------------------------------
# SAFE VALUE CLEANING
# ------------------------------------------

# Mantener temporalmente como string
# porque cada variable tiene formato distinto

df["VALOR_LECTURA"] = (
    df["VALOR_LECTURA"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------
# CATEGORY COLUMNS
# ------------------------------------------

category_cols = [
    "CODI_ESTACIO",
    "CODI_ESTAT",
    "CODI_BASE"
]

for col in category_cols:
    df[col] = df[col].astype("category")

# ------------------------------------------
# RESULTS
# ------------------------------------------

print("\nTipos optimizados:\n")
print(df.dtypes)

print("\nMemoria final:")
print(
    round(df.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB"
)

reduction = (
    1
    - (
        df.memory_usage(deep=True).sum()
        / (545.39 * 1024**2)
    )
) * 100

print(f"\nReducción aproximada: {reduction:.2f}%")

Memoria inicial:
545.39 MB

Tipos optimizados:

ID                          str
CODI_ESTACIO           category
CODI_VARIABLE              int8
DATA_LECTURA     datetime64[us]
DATA_EXTREM                 str
VALOR_LECTURA               str
CODI_ESTAT             category
CODI_BASE              category
dtype: object

Memoria final:
294.57 MB

Reducción aproximada: 45.99%


In [ ]:
# ==========================================
# DROP UNUSED COLUMNS
# ==========================================

df.drop(columns=["DATA_EXTREM"], inplace=True)

print(df.columns)

print("\nMemoria actual:")
print(
    round(df.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB"
)

In [16]:
# ==========================================
# EXPORT TO PARQUET
# ==========================================

PARQUET_OUTPUT = Path("../data/processed/xema_bronze.parquet")

df.to_parquet(
    PARQUET_OUTPUT,
    engine="pyarrow",
    compression="snappy",
    index=False
)

print("✅ Parquet exportado:")
print(PARQUET_OUTPUT)

✅ Parquet exportado:
../data/processed/xema_bronze.parquet


In [17]:
# ==========================================
# LOAD PARQUET
# ==========================================

import time

start = time.time()

df_parquet = pd.read_parquet(
    "../data/processed/xema_bronze.parquet"
)

end = time.time()

print(f"✅ Tiempo carga parquet: {end - start:.2f} segundos")

print("\nShape:")
print(df_parquet.shape)

print("\nDtypes:")
print(df_parquet.dtypes)

print("\nMemoria:")
print(
    round(df_parquet.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB"
)

display(df_parquet.head())

✅ Tiempo carga parquet: 0.11 segundos

Shape:
(4791830, 7)

Dtypes:
ID                          str
CODI_ESTACIO           category
CODI_VARIABLE              int8
DATA_LECTURA     datetime64[us]
VALOR_LECTURA               str
CODI_ESTAT             category
CODI_BASE              category
dtype: object

Memoria:
206.74 MB


,ID,CODI_ESTACIO,CODI_VARIABLE,DATA_LECTURA,VALOR_LECTURA,CODI_ESTAT,CODI_BASE
0,WU320101090000,WU,32,2009-01-01,"10,2",V,SH
1,WU330101090000,WU,33,2009-01-01,89,V,SH
2,WU340101090000,WU,34,2009-01-01,1.016,V,SH
3,WU350101090000,WU,35,2009-01-01,0,V,SH
4,WU010101090000,WU,1,2009-01-01,1.017,V,HO


In [29]:
# ==========================================
# LOAD VARIABLES METADATA
# ==========================================

metadata_file = Path(
    "../data/raw/Metadades_variables_meteorològiques_20260613.csv"
)

variables_df = pd.read_csv(
    metadata_file,
    sep=",",
    low_memory=False
)

print("Shape metadata:")
print(variables_df.shape)

print("\nColumnas:")
print(variables_df.columns.tolist())

print("\nDtypes:")
print(variables_df.dtypes)

display(variables_df.head(10))

Shape metadata:
(67, 6)

Columnas:
['CODI_VARIABLE', 'NOM_VARIABLE', 'UNITAT', 'ACRONIM', 'CODI_TIPUS_VAR', 'DECIMALS']

Dtypes:
CODI_VARIABLE     int64
NOM_VARIABLE        str
UNITAT              str
ACRONIM             str
CODI_TIPUS_VAR      str
DECIMALS          int64
dtype: object


,CODI_VARIABLE,NOM_VARIABLE,UNITAT,ACRONIM,CODI_TIPUS_VAR,DECIMALS
0,2,Pressió atmosfèrica mínima,hPa,Pn,DAT,1
1,3,Humitat relativa màxima,%,HRx,DAT,0
2,30,Velocitat del vent a 10 m (esc.),m/s,VV10,DAT,1
3,31,Direcció de vent 10 m (m. 1),°,DV10,DAT,0
4,32,Temperatura,°C,T,DAT,1
5,33,Humitat relativa,%,HR,DAT,0
6,34,Pressió atmosfèrica,hPa,P,DAT,1
7,35,Precipitació,mm,PPT,DAT,1
8,36,Irradiància solar global,W/m²,RS,DAT,0
9,38,Gruix de neu a terra,cm,GNEU,DAT,0


In [30]:
# ==========================================
# VARIABLE CODE AUDIT
# ==========================================

# Variables presentes en dataset principal
main_vars = set(df_parquet["CODI_VARIABLE"].unique())

# Variables presentes en metadata
meta_vars = set(variables_df["CODI_VARIABLE"].unique())

# Variables sin metadata
missing_vars = sorted(main_vars - meta_vars)

# Variables metadata no usadas
unused_meta = sorted(meta_vars - main_vars)

print("Variables dataset principal:")
print(len(main_vars))

print("\nVariables metadata:")
print(len(meta_vars))

print("\nVariables SIN metadata:")
print(missing_vars)

print("\nVariables metadata NO usadas:")
print(unused_meta)

Variables dataset principal:
16

Variables metadata:
67

Variables SIN metadata:
[np.int8(1)]

Variables metadata NO usadas:
[np.int64(30), np.int64(31), np.int64(38), np.int64(46), np.int64(47), np.int64(50), np.int64(51), np.int64(56), np.int64(57), np.int64(59), np.int64(1000), np.int64(1001), np.int64(1002), np.int64(1003), np.int64(1004), np.int64(1100), np.int64(1101), np.int64(1102), np.int64(1200), np.int64(1201), np.int64(1202), np.int64(1300), np.int64(1301), np.int64(1302), np.int64(1303), np.int64(1304), np.int64(1305), np.int64(1400), np.int64(1401), np.int64(1500), np.int64(1501), np.int64(1502), np.int64(1503), np.int64(1504), np.int64(1505), np.int64(1506), np.int64(1507), np.int64(1508), np.int64(1509), np.int64(1510), np.int64(1511), np.int64(1512), np.int64(1513), np.int64(1514), np.int64(1515), np.int64(1516), np.int64(1517), np.int64(1600), np.int64(1601), np.int64(1602), np.int64(1603), np.int64(1700)]


In [31]:
# ==========================================
# ADD LEGACY VARIABLE
# ==========================================

manual_rows = pd.DataFrame([
    {
        "CODI_VARIABLE": 1,
        "NOM_VARIABLE": "Pressió atmosfèrica (legacy)",
        "UNITAT": "hPa",
        "ACRONIM": "P_LEGACY",
        "CODI_TIPUS_VAR": "DAT",
        "DECIMALS": 0
    }
])

variables_df = pd.concat(
    [variables_df, manual_rows],
    ignore_index=True
)

# ==========================================
# MERGE METADATA
# ==========================================

df_enriched = df_parquet.merge(
    variables_df,
    on="CODI_VARIABLE",
    how="left"
)

print("Shape merged:")
print(df_enriched.shape)

print("\nMissing metadata:")
print(
    df_enriched["NOM_VARIABLE"]
    .isna()
    .sum()
)

display(df_enriched.head())

Shape merged:
(4791830, 12)

Missing metadata:
0


,ID,CODI_ESTACIO,CODI_VARIABLE,DATA_LECTURA,VALOR_LECTURA,CODI_ESTAT,CODI_BASE,NOM_VARIABLE,UNITAT,ACRONIM,CODI_TIPUS_VAR,DECIMALS
0,WU320101090000,WU,32,2009-01-01,"10,2",V,SH,Temperatura,°C,T,DAT,1
1,WU330101090000,WU,33,2009-01-01,89,V,SH,Humitat relativa,%,HR,DAT,0
2,WU340101090000,WU,34,2009-01-01,1.016,V,SH,Pressió atmosfèrica,hPa,P,DAT,1
3,WU350101090000,WU,35,2009-01-01,0,V,SH,Precipitació,mm,PPT,DAT,1
4,WU010101090000,WU,1,2009-01-01,1.017,V,HO,Pressió atmosfèrica (legacy),hPa,P_LEGACY,DAT,0


In [32]:
# ==========================================
# ROBUST VALUE CLEANING
# ==========================================

df_silver = df_enriched.copy()

# Inicializar columna
df_silver["VALOR_CLEAN"] = df_silver["VALOR_LECTURA"].astype(str)

# ------------------------------------------
# VARIABLES SIN DECIMALES
# ------------------------------------------

mask_int = df_silver["DECIMALS"] == 0

df_silver.loc[mask_int, "VALOR_CLEAN"] = (
    df_silver.loc[mask_int, "VALOR_CLEAN"]
    .str.replace(".", "", regex=False)
    .str.replace(",", "", regex=False)
)

# ------------------------------------------
# VARIABLES CON DECIMALES
# ------------------------------------------

mask_float = df_silver["DECIMALS"] > 0

df_silver.loc[mask_float, "VALOR_CLEAN"] = (
    df_silver.loc[mask_float, "VALOR_CLEAN"]
    # eliminar separador miles
    .str.replace(".", "", regex=False)
    # convertir decimal europeo
    .str.replace(",", ".", regex=False)
)

# ------------------------------------------
# CONVERT NUMERIC
# ------------------------------------------

df_silver["VALOR_CLEAN"] = pd.to_numeric(
    df_silver["VALOR_CLEAN"],
    errors="coerce"
)

# ------------------------------------------
# RESULTS
# ------------------------------------------

print("NaN after conversion:")
print(
    df_silver["VALOR_CLEAN"]
    .isna()
    .sum()
)

print("\nDtype:")
print(df_silver["VALOR_CLEAN"].dtype)

display(
    df_silver[
        [
            "NOM_VARIABLE",
            "VALOR_LECTURA",
            "VALOR_CLEAN",
            "DECIMALS"
        ]
    ].head(30)
)

NaN after conversion:
0

Dtype:
float64


,NOM_VARIABLE,VALOR_LECTURA,VALOR_CLEAN,DECIMALS
0,Temperatura,"10,2",10.2,1
1,Humitat relativa,89,89.0,0
2,Pressió atmosfèrica,1.016,1016.0,1
3,Precipitació,0,0.0,1
4,Pressió atmosfèrica (legacy),1.017,1017.0,0
5,Temperatura màxima,"10,4",10.4,1
6,Temperatura mínima,"9,9",9.9,1
7,Humitat relativa mínima,88,88.0,0
8,Ratxa màxima del vent a 6 m,"4,2",4.2,1
9,Irradiància solar global,0,0.0,0


In [33]:
df_silver["VALOR_CLEAN"] = (
    df_silver["VALOR_CLEAN"]
    .astype("float32")
)

In [34]:
# ==========================================
# QUALITY FILTERING
# ==========================================

print("Shape original:")
print(df_silver.shape)

# ------------------------------------------
# KEEP ONLY VALID OBSERVATIONS
# ------------------------------------------

df_silver = df_silver[
    df_silver["CODI_ESTAT"] == "V"
].copy()

print("\nShape después filtro calidad:")
print(df_silver.shape)

# ------------------------------------------
# REMOVE NaN VALUES
# ------------------------------------------

before_nan = len(df_silver)

df_silver = df_silver[
    df_silver["VALOR_CLEAN"].notna()
].copy()

after_nan = len(df_silver)

print("\nFilas eliminadas por NaN:")
print(before_nan - after_nan)

# ------------------------------------------
# REMOVE DUPLICATES
# ------------------------------------------

before_dup = len(df_silver)

df_silver = df_silver.drop_duplicates()

after_dup = len(df_silver)

print("\nDuplicados eliminados:")
print(before_dup - after_dup)

# ------------------------------------------
# FINAL MEMORY OPTIMIZATION
# ------------------------------------------

df_silver["VALOR_CLEAN"] = (
    df_silver["VALOR_CLEAN"]
    .astype("float32")
)

print("\nMemoria final:")
print(
    round(
        df_silver.memory_usage(deep=True).sum()
        / 1024**2,
        2
    ),
    "MB"
)

display(df_silver.head())

Shape original:
(4791830, 13)

Shape después filtro calidad:
(4227737, 13)

Filas eliminadas por NaN:
0

Duplicados eliminados:
0

Memoria final:
529.64 MB


,ID,CODI_ESTACIO,CODI_VARIABLE,DATA_LECTURA,VALOR_LECTURA,CODI_ESTAT,CODI_BASE,NOM_VARIABLE,UNITAT,ACRONIM,CODI_TIPUS_VAR,DECIMALS,VALOR_CLEAN
0,WU320101090000,WU,32,2009-01-01,"10,2",V,SH,Temperatura,°C,T,DAT,1,10.2
1,WU330101090000,WU,33,2009-01-01,89,V,SH,Humitat relativa,%,HR,DAT,0,89.0
2,WU340101090000,WU,34,2009-01-01,1.016,V,SH,Pressió atmosfèrica,hPa,P,DAT,1,1016.0
3,WU350101090000,WU,35,2009-01-01,0,V,SH,Precipitació,mm,PPT,DAT,1,0.0
4,WU010101090000,WU,1,2009-01-01,1.017,V,HO,Pressió atmosfèrica (legacy),hPa,P_LEGACY,DAT,0,1017.0


In [35]:
# ==========================================
# FINAL SILVER OPTIMIZATION
# ==========================================

# ------------------------------------------
# DROP UNUSED COLUMNS
# ------------------------------------------

drop_cols = [
    "VALOR_LECTURA"
]

df_silver.drop(columns=drop_cols, inplace=True)

# ------------------------------------------
# CATEGORY OPTIMIZATION
# ------------------------------------------

category_cols = [
    "CODI_ESTACIO",
    "CODI_ESTAT",
    "CODI_BASE",
    "NOM_VARIABLE",
    "UNITAT",
    "ACRONIM",
    "CODI_TIPUS_VAR"
]

for col in category_cols:
    df_silver[col] = df_silver[col].astype("category")

# ------------------------------------------
# NUMERIC OPTIMIZATION
# ------------------------------------------

df_silver["DECIMALS"] = pd.to_numeric(
    df_silver["DECIMALS"],
    downcast="integer"
)

df_silver["VALOR_CLEAN"] = (
    df_silver["VALOR_CLEAN"]
    .astype("float32")
)

# ------------------------------------------
# RESULTS
# ------------------------------------------

print("Dtypes:\n")
print(df_silver.dtypes)

print("\nMemoria optimizada:")
print(
    round(
        df_silver.memory_usage(deep=True).sum()
        / 1024**2,
        2
    ),
    "MB"
)

Dtypes:

ID                           str
CODI_ESTACIO            category
CODI_VARIABLE               int8
DATA_LECTURA      datetime64[us]
CODI_ESTAT              category
CODI_BASE               category
NOM_VARIABLE            category
UNITAT                  category
ACRONIM                 category
CODI_TIPUS_VAR          category
DECIMALS                    int8
VALOR_CLEAN              float32
dtype: object

Memoria optimizada:
206.13 MB


In [36]:
# ==========================================
# EXPORT SILVER PARQUET
# ==========================================

SILVER_OUTPUT = Path(
    "../data/processed/xema_silver.parquet"
)

df_silver.to_parquet(
    SILVER_OUTPUT,
    engine="pyarrow",
    compression="snappy",
    index=False
)

print("✅ Silver parquet exportado:")
print(SILVER_OUTPUT)

print("\nTamaño final dataframe:")
print(df_silver.shape)

print("\nMemoria dataframe:")
print(
    round(
        df_silver.memory_usage(deep=True).sum()
        / 1024**2,
        2
    ),
    "MB"
)

✅ Silver parquet exportado:
../data/processed/xema_silver.parquet

Tamaño final dataframe:
(4227737, 12)

Memoria dataframe:
206.13 MB
